# Matrix Factorisation — Film Öneri Sistemi

Elimdeki 20 kullanıcı × 1000 film matrisi oldukça seyrek. Yani kullanıcıların çoğu filmlerin çok azına puan vermiş. Bu yüzden tüm matrisi doğrudan tahmin etmek yerine matrisi iki küçük matrise ayırarak yaklaşık olarak tahmin etmeye çalıştım. Bunun için kullanıcı matrisi `u` ve film matrisi `v` kullandım.

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [3]:
from pathlib import Path

data_path = Path("ratings_long.csv")
if not data_path.exists():
    data_path = Path("../ratings_long.csv")

df = pd.read_csv(data_path)
df.head()

,userId,movieId,rating
0,0,16,5
1,0,72,5
2,0,86,5
3,0,259,1
4,0,319,4


In [5]:
r = np.full((20, 1000), fill_value=np.nan)

for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Derecelendirme matrisi $r$, boyutu $20 \times 1000$ olmasına rağmen %1'den az dolu — veri son derece seyrek.

Bunu doğrudan öğrenmek yerine şu ayrışımı kullandım:
- **u** = $20 \times 4$ → kullanıcı gizli özellik matrisi
- **v** = $4 \times 1000$ → film gizli özellik matrisi
- $r \approx u \times v$

In [7]:
print("Rating matrix shape:", r.shape)
print("Observed rating count:", np.sum(~np.isnan(r)))

Rating matrix shape: (20, 1000)
Observed rating count: 200


## Problem & Kullandığım Teknikler

**1. Kayıp Fonksiyonu — MSE + L2 Regularizasyon**  
Bu çalışmada kayıp fonksiyonu olarak sadece gözlemlenen rating değerleri üzerinden MSE kullandım. Ayrıca overfitting'i azaltmak için L2 regularization ekledim. Matrix factorization probleminde `u` ve `v` birlikte öğrenildiği için problem tamamen konveks sayılmaz, ama gradient descent ile kaybı adım adım azaltmaya çalıştım.

**2. Optimizasyon — Gradient Descent**  
`u` ve `v`'yi analitik çözmek yerine her adımda gradyan yönünde küçük adımlar attım. Gradyan normunu izleyerek yeterince küçüldüğünde erken durdurdum (early stopping).

**3. Regularizasyon — L2 (Frobenius Norm)**  
`u` ve `v` matrislerinin elemanlarının karelerini kayba ekledim. L1'e göre türevi her yerde tanımlı olduğundan gradyan hesabını kolaylaştırıyor.

### Gradient Descent için Hazırlık

Gradyan inişinde sadece bilinen derecelendirmeler üzerinden hata hesaplamak istiyorum. Bu yüzden bir `mask` matrisi kullandım; gözlemlenmiş girişler 1, bilinmeyenler 0 oluyor. `r0`'da NaN'ları 0 ile doldurdum çünkü matris çarpmalarında NaN yayılmasın istiyorum — zaten `mask` o konumları sıfırladığı için kayba katkıları olmuyor.

In [8]:
mask = (~np.isnan(r)).astype(float)
N    = int(mask.sum())
r0   = np.nan_to_num(r, nan=0.0)

$$L(u,v) = \frac{1}{N} \sum_{(i,j)\in\Omega} \bigl(r_{ij} - (u \cdot v)_{ij}\bigr)^2 + \lambda\bigl(\|u\|_F^2 + \|v\|_F^2\bigr)$$

$$\frac{\partial L}{\partial u} = -\frac{2}{N}\,\text{err}\cdot v^T + 2\lambda u \qquad \frac{\partial L}{\partial v} = -\frac{2}{N}\,u^T\cdot\text{err} + 2\lambda v$$

In [9]:
def mf_loss(r0, u, v, mask, N, lamb):
    err = (r0 - u @ v) * mask
    return (err ** 2).sum() / N + lamb * (np.sum(u ** 2) + np.sum(v ** 2))

In [10]:
def matrix_factorization(r0, mask, N, k=4, n_epoch=1000, lr=0.5, lamb=0.01, early_stop=True):
    n_users, n_items = r0.shape
    u = np.random.randn(n_users, k) * 0.1
    v = np.random.randn(k, n_items) * 0.1
    loss_history = []

    for epoch in range(n_epoch):
        r_hat = u @ v
        err   = (r0 - r_hat) * mask

        g_u = -(2 / N) * (err @ v.T) + 2 * lamb * u
        g_v = -(2 / N) * (u.T @ err) + 2 * lamb * v

        grad_norm = np.linalg.norm(g_u) + np.linalg.norm(g_v)
        loss      = mf_loss(r0, u, v, mask, N, lamb)
        loss_history.append(loss)

        if epoch % 50 == 0:
            print(f"Epoch {epoch:4d} | grad_norm: {grad_norm:.6f} | loss: {loss:.4f}")

        if grad_norm < 1e-3 and early_stop:
            break

        u -= lr * g_u
        v -= lr * g_v

    return u, v, loss_history

In [11]:
np.random.seed(42)
u, v, loss_history = matrix_factorization(r0, mask, N)

Epoch    0 | grad_norm: 0.265470 | loss: 11.4790
Epoch   50 | grad_norm: 0.749068 | loss: 8.5321
Epoch  100 | grad_norm: 0.306175 | loss: 4.1968
Epoch  150 | grad_norm: 0.090959 | loss: 3.6700
Epoch  200 | grad_norm: 0.034268 | loss: 3.6286
Epoch  250 | grad_norm: 0.024676 | loss: 3.6176
Epoch  300 | grad_norm: 0.020633 | loss: 3.6111
Epoch  350 | grad_norm: 0.018144 | loss: 3.6063
Epoch  400 | grad_norm: 0.016453 | loss: 3.6026
Epoch  450 | grad_norm: 0.015201 | loss: 3.5994
Epoch  500 | grad_norm: 0.014206 | loss: 3.5967
Epoch  550 | grad_norm: 0.013380 | loss: 3.5943
Epoch  600 | grad_norm: 0.012672 | loss: 3.5922
Epoch  650 | grad_norm: 0.012056 | loss: 3.5903
Epoch  700 | grad_norm: 0.011511 | loss: 3.5886
Epoch  750 | grad_norm: 0.011024 | loss: 3.5870
Epoch  800 | grad_norm: 0.010587 | loss: 3.5855
Epoch  850 | grad_norm: 0.010190 | loss: 3.5842
Epoch  900 | grad_norm: 0.009828 | loss: 3.5829
Epoch  950 | grad_norm: 0.009495 | loss: 3.5818


In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=loss_history, mode="lines", name="Training Loss"))
fig.update_layout(title="Matrix Factorisation Loss", xaxis_title="Epoch", yaxis_title="Loss")
fig.show()

## Model Performansı

Modeli eğitimde kullandığım gözlemler üzerinde ölçüm yaptım. MAE ve RMSE'yi ikisini birden yazdırdım çünkü RMSE büyük hataları daha çok cezalandırıyor; ikisi arasındaki fark bana hata dağılımı hakkında fikir veriyor.

In [12]:
obs    = mask.astype(bool)
r_pred = (u @ v)[obs]

mae  = np.mean(np.abs(r[obs] - r_pred))
rmse = np.sqrt(np.mean((r[obs] - r_pred) ** 2))
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

MAE  : 0.5539
RMSE : 0.6181
